# Mel vs inverted-Mel vs LFCC — EER table

Eval each mixed-trained checkpoint on **ASVspoof2017** and **PA2019** (dev by default).

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "experiment_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare")
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from experiment_lib import (
    FEATURE_TYPES,
    RUNS_DIR,
    build_comparison_table,
    default_checkpoint,
    eval_checkpoint_on_corpora,
    markdown_eer_table,
)

ASV17_SPLIT = "dev"
PA_SPLIT = "dev"
BATCH_SIZE = 8
FORCE_CPU = False

In [ ]:
all_results = []
for ft in FEATURE_TYPES:
    ckpt = default_checkpoint(ft)
    if not ckpt.exists():
        print("MISSING", ckpt)
        continue
    print("\nEval", ft, "<-", ckpt)
    out = RUNS_DIR / "eval" / ft
    res = eval_checkpoint_on_corpora(
        ckpt,
        asv17_split=ASV17_SPLIT,
        pa_split=PA_SPLIT,
        batch_size=BATCH_SIZE,
        force_cpu=FORCE_CPU,
        output_dir=out,
    )
    all_results.append(res)
    print(
        f"  2017 EER={res['asvspoof2017']['eer_percent']:.2f}%"
        f"  PA EER={res['pa2019']['eer_percent']:.2f}%"
    )


Eval mel <- D:\speaker-verification-system\replay-cnn-baseline\experiments\lfcc_vs_mel_compare\runs\mel\best_mel_mixed_2017_pa2019.pt


Checking dev audio:   5%|▍         | 1393/29700 [00:11<04:44, 99.50it/s] 

## Comparison

In [ ]:
rows = build_comparison_table(all_results)
md = markdown_eer_table(rows)
display(Markdown(md))
print(md)

out_json = RUNS_DIR / "eval" / "comparison_table.json"
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(json.dumps({"rows": rows, "results": all_results}, indent=2), encoding="utf-8")
print("Wrote", out_json)